# Práctica 3: Soluciones de los ejercicios propuestos
# Inteligencia Artificial
# Grado en Ingeniería Informática - Ingeniería del Software
# Universidad de Sevilla

Los ejercicios que se plantean a continuación tienen como objetivo el practicar con la biblioteca [NLTK](https://www.nltk.org) de Python.

### Ejercicio 1

El objetivo de este ejercicio es entrenar y evaluar el rendimiento de un filtro de correo electrónico no deseado. Para ello se usará el corpus Enron-Spam, pero no se proporcionará un vocabulario fijo, sino que este deberá aprenderse a partir de los mensajes de entrenamiento. Con el objetivo de homogeneizar el vocabulario aprendido y de mejorar el rendimiento del filtro construido, se pedirá que se apliquen distintas técnicas de preprocesado.

En todos los apartados de este ejercicio se deberá realizar lo siguiente:

* Construir el filtro como una tubería de scikit-learn que concatene un vectorizador tf-idf y un modelo $k$NN clasificador con 5 vecinos y que use la métrica del coseno.
* Definir una función `procesa_mensaje` que, dado el contenido en bruto de un mensaje, aplique todos los pasos de procesamiento pedidos hasta obtener la lista de tókenes correspondiente. Esta función se deberá proporcionar como argumento `analyzer` del vectorizador tf-idf.
* Entrenar el filtro con el corpus de entrenamiento.
* Calcular la sensibilidad del filtro sobre el corpus de prueba.

In [1]:
from email import parser
from email import policy

In [2]:
analizador_mensaje = parser.Parser(policy=policy.default)

In [3]:
from pathlib import Path

In [4]:
carpeta_Enron_Spam = Path('Filtro antispam/Enron-Spam/')
carpeta_entrenamiento = carpeta_Enron_Spam / 'train'
carpeta_prueba = carpeta_Enron_Spam / 'test'

contenidos_mensajes_entrenamiento = []
clases_mensajes_entrenamiento = []
for ruta_mensaje in (carpeta_entrenamiento / 'legítimo').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_entrenamiento.append(mensaje.get_content())
            clases_mensajes_entrenamiento.append(0)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass
for ruta_mensaje in (carpeta_entrenamiento / 'no_deseado').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_entrenamiento.append(mensaje.get_content())
            clases_mensajes_entrenamiento.append(1)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass

contenidos_mensajes_prueba = []
clases_mensajes_prueba = []
for ruta_mensaje in (carpeta_prueba / 'legítimo').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_prueba.append(mensaje.get_content())
            clases_mensajes_prueba.append(0)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass
for ruta_mensaje in (carpeta_prueba / 'no_deseado').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_prueba.append(mensaje.get_content())
            clases_mensajes_prueba.append(1)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass

#### Apartado 0

En este apartado se pide procesar los mensajes realizando los siguientes 3 pasos:

* Extraer el contenido de texto de los mensajes en formato HTML. Para ello hacer uso de la biblioteca [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/).
* Dividir el contenido de los mensajes en secuencias de tókenes mediante el tokenizador de NLTK.
* Eliminar de los tókenes los caracteres no alfanuméricos (y eliminar por completo aquellos tókenes que no contengan caracteres alfanuméricos).

In [5]:
contenidos_mensajes_entrenamiento[-21]

'"I just wanted to write and thank you for Spur-M. \nI suffered from poor sperm count and motility. I found \nyour site and ordered Spur-M Fertility Blend for Men. \nI have wondered for years what caused low semen and sperm \ncount, and how I could improve my fertility and help my wife\nconceive. Spur-M seems to have done just that! Thank you\nfor your support."\nAndrew H., London, UK\n\n"Spur-M really does help improve fertility and effectiveness\nof sperm and semen motility. I used it for the past few months,\nand not only does it work - I also feel better to. I have \nmore energy. This is an excellent counter to low sperm count\nand motility. I\'ll be buying more!!!"\nFranz K., Bonn, Germany\n\n"I had been wondering on the causes of low semen and \nsperm count, I was searching for this type of information \nwhen I found your site. I hadn\'t been made aware of this \nproduct before then, so was quite surprised to be able \nto find a Male fertility product. Usually everything is \ngea

In [6]:
from bs4 import BeautifulSoup

In [7]:
def elimina_html(contenido):
    return BeautifulSoup(contenido).get_text()

In [8]:
elimina_html(contenidos_mensajes_entrenamiento[-21])

'"I just wanted to write and thank you for Spur-M. \nI suffered from poor sperm count and motility. I found \nyour site and ordered Spur-M Fertility Blend for Men. \nI have wondered for years what caused low semen and sperm \ncount, and how I could improve my fertility and help my wife\nconceive. Spur-M seems to have done just that! Thank you\nfor your support."\nAndrew H., London, UK\n\n"Spur-M really does help improve fertility and effectiveness\nof sperm and semen motility. I used it for the past few months,\nand not only does it work - I also feel better to. I have \nmore energy. This is an excellent counter to low sperm count\nand motility. I\'ll be buying more!!!"\nFranz K., Bonn, Germany\n\n"I had been wondering on the causes of low semen and \nsperm count, I was searching for this type of information \nwhen I found your site. I hadn\'t been made aware of this \nproduct before then, so was quite surprised to be able \nto find a Male fertility product. Usually everything is \ngea

In [9]:
import os

os.environ['NLTK_DATA'] = '.'

from nltk import download

download('punkt', download_dir='.')

download('punkt_tab', download_dir='.')

[nltk_data] Downloading package punkt to ....
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to ....
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [10]:
from nltk.tokenize import word_tokenize

In [11]:
from pprint import pprint

In [12]:
elimina_html(contenidos_mensajes_entrenamiento[-21])

'"I just wanted to write and thank you for Spur-M. \nI suffered from poor sperm count and motility. I found \nyour site and ordered Spur-M Fertility Blend for Men. \nI have wondered for years what caused low semen and sperm \ncount, and how I could improve my fertility and help my wife\nconceive. Spur-M seems to have done just that! Thank you\nfor your support."\nAndrew H., London, UK\n\n"Spur-M really does help improve fertility and effectiveness\nof sperm and semen motility. I used it for the past few months,\nand not only does it work - I also feel better to. I have \nmore energy. This is an excellent counter to low sperm count\nand motility. I\'ll be buying more!!!"\nFranz K., Bonn, Germany\n\n"I had been wondering on the causes of low semen and \nsperm count, I was searching for this type of information \nwhen I found your site. I hadn\'t been made aware of this \nproduct before then, so was quite surprised to be able \nto find a Male fertility product. Usually everything is \ngea

In [13]:
word_tokenize(elimina_html(contenidos_mensajes_entrenamiento[-21]))

['``',
 'I',
 'just',
 'wanted',
 'to',
 'write',
 'and',
 'thank',
 'you',
 'for',
 'Spur-M',
 '.',
 'I',
 'suffered',
 'from',
 'poor',
 'sperm',
 'count',
 'and',
 'motility',
 '.',
 'I',
 'found',
 'your',
 'site',
 'and',
 'ordered',
 'Spur-M',
 'Fertility',
 'Blend',
 'for',
 'Men',
 '.',
 'I',
 'have',
 'wondered',
 'for',
 'years',
 'what',
 'caused',
 'low',
 'semen',
 'and',
 'sperm',
 'count',
 ',',
 'and',
 'how',
 'I',
 'could',
 'improve',
 'my',
 'fertility',
 'and',
 'help',
 'my',
 'wife',
 'conceive',
 '.',
 'Spur-M',
 'seems',
 'to',
 'have',
 'done',
 'just',
 'that',
 '!',
 'Thank',
 'you',
 'for',
 'your',
 'support',
 '.',
 "''",
 'Andrew',
 'H.',
 ',',
 'London',
 ',',
 'UK',
 "''",
 'Spur-M',
 'really',
 'does',
 'help',
 'improve',
 'fertility',
 'and',
 'effectiveness',
 'of',
 'sperm',
 'and',
 'semen',
 'motility',
 '.',
 'I',
 'used',
 'it',
 'for',
 'the',
 'past',
 'few',
 'months',
 ',',
 'and',
 'not',
 'only',
 'does',
 'it',
 'work',
 '-',
 'I',
 'al

In [14]:
pprint(word_tokenize(elimina_html(contenidos_mensajes_entrenamiento[-21])),
       compact=True)

['``', 'I', 'just', 'wanted', 'to', 'write', 'and', 'thank', 'you', 'for',
 'Spur-M', '.', 'I', 'suffered', 'from', 'poor', 'sperm', 'count', 'and',
 'motility', '.', 'I', 'found', 'your', 'site', 'and', 'ordered', 'Spur-M',
 'Fertility', 'Blend', 'for', 'Men', '.', 'I', 'have', 'wondered', 'for',
 'years', 'what', 'caused', 'low', 'semen', 'and', 'sperm', 'count', ',', 'and',
 'how', 'I', 'could', 'improve', 'my', 'fertility', 'and', 'help', 'my', 'wife',
 'conceive', '.', 'Spur-M', 'seems', 'to', 'have', 'done', 'just', 'that', '!',
 'Thank', 'you', 'for', 'your', 'support', '.', "''", 'Andrew', 'H.', ',',
 'London', ',', 'UK', "''", 'Spur-M', 'really', 'does', 'help', 'improve',
 'fertility', 'and', 'effectiveness', 'of', 'sperm', 'and', 'semen', 'motility',
 '.', 'I', 'used', 'it', 'for', 'the', 'past', 'few', 'months', ',', 'and',
 'not', 'only', 'does', 'it', 'work', '-', 'I', 'also', 'feel', 'better', 'to',
 '.', 'I', 'have', 'more', 'energy', '.', 'This', 'is', 'an', 'excellent

La eliminación de los caracteres no alfanuméricos se puede realizar mediante expresiones regulares, usando para ello el paquete [re](https://docs.python.org/es/3/library/re.html) de la biblioteca estándar de Python.

In [15]:
import re

In [16]:
def elimina_no_alfanumerico(contenido):
    return [re.sub(r'[^\w]', '', palabra)
            for palabra in contenido
            if re.search(r'\w', palabra)]

In [17]:
def procesa_mensaje(contenido):
    contenido = elimina_html(contenido)
    contenido = word_tokenize(contenido)
    contenido = elimina_no_alfanumerico(contenido)
    return contenido

In [18]:
pprint(procesa_mensaje(contenidos_mensajes_entrenamiento[-21]),
       compact=True)

['I', 'just', 'wanted', 'to', 'write', 'and', 'thank', 'you', 'for', 'SpurM',
 'I', 'suffered', 'from', 'poor', 'sperm', 'count', 'and', 'motility', 'I',
 'found', 'your', 'site', 'and', 'ordered', 'SpurM', 'Fertility', 'Blend',
 'for', 'Men', 'I', 'have', 'wondered', 'for', 'years', 'what', 'caused', 'low',
 'semen', 'and', 'sperm', 'count', 'and', 'how', 'I', 'could', 'improve', 'my',
 'fertility', 'and', 'help', 'my', 'wife', 'conceive', 'SpurM', 'seems', 'to',
 'have', 'done', 'just', 'that', 'Thank', 'you', 'for', 'your', 'support',
 'Andrew', 'H', 'London', 'UK', 'SpurM', 'really', 'does', 'help', 'improve',
 'fertility', 'and', 'effectiveness', 'of', 'sperm', 'and', 'semen', 'motility',
 'I', 'used', 'it', 'for', 'the', 'past', 'few', 'months', 'and', 'not', 'only',
 'does', 'it', 'work', 'I', 'also', 'feel', 'better', 'to', 'I', 'have', 'more',
 'energy', 'This', 'is', 'an', 'excellent', 'counter', 'to', 'low', 'sperm',
 'count', 'and', 'motility', 'I', 'll', 'be', 'buying', 'm

Debido a la naturaleza de los mensajes no deseados, algunos de ellos pueden confundir a la biblioteca Beautiful Soup, avisando esta de que el mensaje puede tratarse de una URL o de una ruta a un fichero, en lugar de un mensaje de correo electrónico. El código de la siguiente celda filtra ese tipo de avisos.

In [19]:
from bs4 import MarkupResemblesLocatorWarning
import warnings

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

Estamos ya en condiciones de poder construir el filtro de correo electrónico no deseado.

In [20]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier

In [21]:
# probar el vectorizador
vectorizador = TfidfVectorizer(analyzer=procesa_mensaje)
vectorizador.fit(contenidos_mensajes_entrenamiento)

TfidfVectorizer(analyzer=<function procesa_mensaje at 0x000001E548C62F20>)

In [22]:
# total de rasgos
len(vectorizador.get_feature_names_out())

170490

In [23]:
contenidos_mensajes_entrenamiento[-21]

'"I just wanted to write and thank you for Spur-M. \nI suffered from poor sperm count and motility. I found \nyour site and ordered Spur-M Fertility Blend for Men. \nI have wondered for years what caused low semen and sperm \ncount, and how I could improve my fertility and help my wife\nconceive. Spur-M seems to have done just that! Thank you\nfor your support."\nAndrew H., London, UK\n\n"Spur-M really does help improve fertility and effectiveness\nof sperm and semen motility. I used it for the past few months,\nand not only does it work - I also feel better to. I have \nmore energy. This is an excellent counter to low sperm count\nand motility. I\'ll be buying more!!!"\nFranz K., Bonn, Germany\n\n"I had been wondering on the causes of low semen and \nsperm count, I was searching for this type of information \nwhen I found your site. I hadn\'t been made aware of this \nproduct before then, so was quite surprised to be able \nto find a Male fertility product. Usually everything is \ngea

In [24]:
# obtener la representacion de 1 documento
tfidf = vectorizador.transform([contenidos_mensajes_entrenamiento[-21]])

In [25]:
# valores internos de tfidf
print(tfidf)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 150 stored elements and shape (1, 170490)>
  Coords	Values
  (0, 2959)	0.03445671586938173
  (0, 27026)	0.05448231494665655
  (0, 28132)	0.043766018693933414
  (0, 30130)	0.07328603641788871
  (0, 30351)	0.07524274714906704
  (0, 31243)	0.04288179789584623
  (0, 38516)	0.06967961382947553
  (0, 41773)	0.08015633593552479
  (0, 43362)	0.07580412785456886
  (0, 44093)	0.07244364269237369
  (0, 45424)	0.05865115177432897
  (0, 46429)	0.04537099712284663
  (0, 48866)	0.28921748795967034
  (0, 50726)	0.027912582870160414
  (0, 51852)	0.04953474466998969
  (0, 55023)	0.04332202446246226
  (0, 56734)	0.05991361983720034
  (0, 57892)	0.06149844339452558
  (0, 59247)	0.09505964359622003
  (0, 59332)	0.037605775683870656
  (0, 68849)	0.09505964359622003
  (0, 69407)	0.06967961382947553
  (0, 74438)	0.32534828510773006
  (0, 75193)	0.07772685709553807
  (0, 77131)	0.03355057656848707
  :	:
  (0, 154013)	0.03662172872266122
  (0, 154062

In [26]:
# vamos a buscar las palabras que corresponden a cada posición, e imprimir el valor de tfidf para cada una de ellas
palabras = vectorizador.get_feature_names_out()

indices_no_cero = tfidf.nonzero()[1]

for i in indices_no_cero:
    print(f'{palabras[i]}: {tfidf[0, i]}')

100: 0.03445671586938173
Andrew: 0.05448231494665655
B: 0.043766018693933414
Blend: 0.07328603641788871
Bonn: 0.07524274714906704
But: 0.04288179789584623
Doctors: 0.06967961382947553
Essex: 0.08015633593552479
Fertility: 0.07580412785456886
Franz: 0.07244364269237369
Germany: 0.05865115177432897
H: 0.04537099712284663
I: 0.28921748795967034
It: 0.027912582870160414
K: 0.04953474466998969
London: 0.04332202446246226
Male: 0.05991361983720034
Men: 0.06149844339452558
Munozprovencauxnetrmphp: 0.09505964359622003
My: 0.037605775683870656
Richardprovencauxnetspur: 0.09505964359622003
Roy: 0.06967961382947553
SpurM: 0.32534828510773006
Suffice: 0.07772685709553807
Thank: 0.03355057656848707
Thanks: 0.02436609541454751
This: 0.023562427309591932
UK: 0.09659650245273176
Usually: 0.07772685709553807
a: 0.028714284582411116
able: 0.07221294602545723
also: 0.02798841692435256
am: 0.027929015786079252
an: 0.02209257650998211
and: 0.16100920783331407
any: 0.02159975575353641
aware: 0.0435324868330

In [27]:
# buscar la aparicion de "00" en contenidos_mensajes_entrenamiento[-21]
# buscar la seccion donde aparezca, imprimir 30 caracteres hacia atras y adelante

mensaje = contenidos_mensajes_entrenamiento[-21]
patron = "00"

# Buscar todas las ocurrencias de "00"
indice = 0
ocurrencias = []

while indice < len(mensaje):
    posicion = mensaje.find(patron, indice)
    if posicion == -1:
        break
    ocurrencias.append(posicion)
    indice = posicion + 1

# Imprimir el contexto de cada ocurrencia
print(f"Se encontraron {len(ocurrencias)} ocurrencias de '{patron}':\n")
for i, pos in enumerate(ocurrencias, 1):
    inicio = max(0, pos - 60)
    fin = min(len(mensaje), pos + len(patron) + 60)
    contexto = mensaje[inicio:fin]
    print(f"Ocurrencia {i} (posición {pos}):")
    print(f"...{contexto}...")
    print()

Se encontraron 1 ocurrencias de '00':

Ocurrencia 1 (posición 1134):
...d news from the 
Doctors - My wife is pregnant. I can't be 100% sure if 
it was Spur-M that helped. But I am happy enough ...



In [28]:
# mostrar las 10 palabras con mayor valor de tfidf

palabras_con_valores = [(palabras[i], tfidf[0, i]) for i in indices_no_cero]

palabras_ordenadas = sorted(palabras_con_valores, key=lambda x: x[1], reverse=True)

print("Las 10 palabras con mayor valor TF-IDF:\n")
for palabra, valor in palabras_ordenadas[:10]:
    print(f'{palabra}: {valor:.6f}')

Las 10 palabras con mayor valor TF-IDF:

SpurM: 0.325348
fertility: 0.325348
sperm: 0.317824
I: 0.289217
count: 0.220568
motility: 0.206473
semen: 0.202661
and: 0.161009
low: 0.124463
to: 0.123987


In [29]:
filtro_antispam = Pipeline([
    ('vectorizador', TfidfVectorizer(analyzer=procesa_mensaje)),
    ('modelo', KNeighborsClassifier(n_neighbors=5, metric='cosine'))
])

In [30]:
filtro_antispam.fit(contenidos_mensajes_entrenamiento,
                    clases_mensajes_entrenamiento)

Pipeline(steps=[('vectorizador',
                 TfidfVectorizer(analyzer=<function procesa_mensaje at 0x000001E548C62F20>)),
                ('modelo', KNeighborsClassifier(metric='cosine'))])

In [31]:
from sklearn.metrics import recall_score

In [32]:
predicciones_mensajes_prueba = filtro_antispam.predict(
    contenidos_mensajes_prueba
)
recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

0.9391714735186156

In [33]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)

print("=" * 60)
print("RESUMEN DE EVALUACIÓN DEL FILTRO ANTISPAM")
print("=" * 60)

# Métricas individuales
accuracy = accuracy_score(clases_mensajes_prueba, predicciones_mensajes_prueba)
precision = precision_score(clases_mensajes_prueba, predicciones_mensajes_prueba)
recall = recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)
f1 = f1_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

print(f"\nMÉTRICAS DE CLASIFICACIÓN:")
print(f"  Exactitud (Accuracy):   {accuracy:.4f}")
print(f"  Precisión (Precision):  {precision:.4f}")
print(f"  Sensibilidad (Recall):  {recall:.4f}")
print(f"  F1-Score:               {f1:.4f}")

# Matriz de confusión
print(f"\nMATRIZ DE CONFUSIÓN:")
cm = confusion_matrix(clases_mensajes_prueba, predicciones_mensajes_prueba)
print(f"                    Predicho: Legítimo  Predicho: Spam")
print(f"  Real: Legítimo           {cm[0][0]:6d}          {cm[0][1]:6d}")
print(f"  Real: Spam               {cm[1][0]:6d}          {cm[1][1]:6d}")

# Reporte de clasificación completo
print(f"\nREPORTE DETALLADO:")
print(classification_report(clases_mensajes_prueba, predicciones_mensajes_prueba, 
                           target_names=['Legítimo', 'No deseado']))

print("=" * 60)

RESUMEN DE EVALUACIÓN DEL FILTRO ANTISPAM

MÉTRICAS DE CLASIFICACIÓN:
  Exactitud (Accuracy):   0.9699
  Precisión (Precision):  0.9766
  Sensibilidad (Recall):  0.9392
  F1-Score:               0.9575

MATRIZ DE CONFUSIÓN:
                    Predicho: Legítimo  Predicho: Spam
  Real: Legítimo             3341              43
  Real: Spam                  116            1791

REPORTE DETALLADO:
              precision    recall  f1-score   support

    Legítimo       0.97      0.99      0.98      3384
  No deseado       0.98      0.94      0.96      1907

    accuracy                           0.97      5291
   macro avg       0.97      0.96      0.97      5291
weighted avg       0.97      0.97      0.97      5291



#### Apartado 1

En este apartado se pide incorporar al procesado de mensajes los siguientes 2 pasos:

* Expandir las contracciones típicas del idioma inglés. Usar para ello el paquete [contractions](https://github.com/kootenpv/contractions).
* Convertir todos los caracteres a minúsculas.

In [34]:
# El paquete contractions posiblemente deba ser instalado, por ejemplo ejecutando en una celda
# !pip install contractions

import contractions

In [35]:
def expande_contracciones(contenido):
    return contractions.fix(contenido)

In [36]:
def convierte_a_minusculas(contenido):
    return contenido.lower()

In [37]:
def procesa_mensaje(contenido):
    contenido = elimina_html(contenido)
    contenido = expande_contracciones(contenido)
    contenido = convierte_a_minusculas(contenido)
    contenido = word_tokenize(contenido)
    contenido = elimina_no_alfanumerico(contenido)
    return contenido

In [38]:
pprint(procesa_mensaje(contenidos_mensajes_entrenamiento[-21]),
       compact=True)

['i', 'just', 'wanted', 'to', 'write', 'and', 'thank', 'you', 'for', 'spurm',
 'i', 'suffered', 'from', 'poor', 'sperm', 'count', 'and', 'motility', 'i',
 'found', 'your', 'site', 'and', 'ordered', 'spurm', 'fertility', 'blend',
 'for', 'men', 'i', 'have', 'wondered', 'for', 'years', 'what', 'caused', 'low',
 'semen', 'and', 'sperm', 'count', 'and', 'how', 'i', 'could', 'improve', 'my',
 'fertility', 'and', 'help', 'my', 'wife', 'conceive', 'spurm', 'seems', 'to',
 'have', 'done', 'just', 'that', 'thank', 'you', 'for', 'your', 'support',
 'andrew', 'h', 'london', 'uk', 'spurm', 'really', 'does', 'help', 'improve',
 'fertility', 'and', 'effectiveness', 'of', 'sperm', 'and', 'semen', 'motility',
 'i', 'used', 'it', 'for', 'the', 'past', 'few', 'months', 'and', 'not', 'only',
 'does', 'it', 'work', 'i', 'also', 'feel', 'better', 'to', 'i', 'have', 'more',
 'energy', 'this', 'is', 'an', 'excellent', 'counter', 'to', 'low', 'sperm',
 'count', 'and', 'motility', 'i', 'will', 'be', 'buying', 

In [39]:
filtro_antispam = Pipeline([
    ('vectorizador', TfidfVectorizer(analyzer=procesa_mensaje)),
    ('modelo', KNeighborsClassifier(n_neighbors=5, metric='cosine'))
])

In [40]:
filtro_antispam.fit(contenidos_mensajes_entrenamiento,
                    clases_mensajes_entrenamiento)

IndexError: string index out of range

In [ ]:
predicciones_mensajes_prueba = filtro_antispam.predict(
    contenidos_mensajes_prueba
)
recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

In [ ]:
predicciones_mensajes_prueba

#### Apartado 2

Palabras vacías (_stop words_, en inglés) es el nombre que reciben las palabras tales como artículos, pronombres y preposiciones que se considera que no aportan significado para un sistema de procesamiento del lenguaje natural y que, por tanto, deben eliminarse durante las operaciones de preprocesado de texto. El conjunto adecuado de palabras vacías a usar depende del sistema concreto que se esté construyendo, e incluso puede resultar conveniente no hacer uso de esta técnica.

NLTK provee de conjuntos genéricos de palabras vacías para distintos idiomas.

In [ ]:
download('stopwords', download_dir='.')

In [ ]:
from nltk.corpus import stopwords
from nltk.data import path
path.append(".")

In [ ]:
palabras_vacias_ingles = stopwords.words('english')
pprint(palabras_vacias_ingles, compact=True)

En este apartado se pide incorporar al procesado de mensajes la eliminación de palabras vacías.

In [ ]:
def elimina_palabras_vacias(contenido):
    return [palabra
            for palabra in contenido
            if palabra not in palabras_vacias_ingles]

In [ ]:
def procesa_mensaje(contenido):
    contenido = elimina_html(contenido)
    contenido = expande_contracciones(contenido)
    contenido = convierte_a_minusculas(contenido)
    contenido = word_tokenize(contenido)
    contenido = elimina_no_alfanumerico(contenido)
    contenido = elimina_palabras_vacias(contenido)
    return contenido

In [ ]:
pprint(procesa_mensaje(contenidos_mensajes_entrenamiento[-21]),
       compact=True)

In [ ]:
filtro_antispam = Pipeline([
    ('vectorizador', TfidfVectorizer(analyzer=procesa_mensaje)),
    ('modelo', KNeighborsClassifier(n_neighbors=5, metric='cosine'))
])

In [ ]:
filtro_antispam.fit(contenidos_mensajes_entrenamiento,
                    clases_mensajes_entrenamiento)

In [ ]:
predicciones_mensajes_prueba = filtro_antispam.predict(
    contenidos_mensajes_prueba
)
recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

#### Apartado 3

Por razones gramaticales, en un documento de texto van a aparecer con seguridad diferentes formas de una palabra, como organizar, organiza y organizando. Además, existen familias de palabras relacionadas derivativamente con significados similares, como democracia, democrático y democratización. En muchas situaciones, parece que sería útil reducir esos conjuntos de palabras a una raíz común. Para ello se suelen usar los procedimientos de _stemming_ y lematización.

_Stemming_ generalmente se refiere a un proceso heurístico rudimentario que corta los extremos de las palabras con la esperanza de lograr el objetivo correctamente la mayor parte del tiempo y, a menudo, incluye la eliminación de afijos derivativos. La lematización generalmente se refiere a hacer las cosas correctamente con el uso de un vocabulario y análisis morfológico de las palabras, normalmente con el objetivo de eliminar únicamente las terminaciones flexivas y devolver la forma base o de diccionario de una palabra, lo que se conoce como lema.

NLTK provee de varios algoritmos de _stemming_ y lematización. En este apartado se pide incorporar al procesado de mensajes el procedimiento de _stemming_ mediante el [algoritmo de Lancaster](https://www.nltk.org/api/nltk.stem.lancaster.html).

In [ ]:
from nltk.stem.lancaster import LancasterStemmer

In [ ]:
def reduce_a_raiz(contenido):
    reductor = LancasterStemmer()
    return [reductor.stem(palabra)
            for palabra in contenido]

In [ ]:
def procesa_mensaje(contenido):
    contenido = elimina_html(contenido)
    contenido = expande_contracciones(contenido)
    contenido = convierte_a_minusculas(contenido)
    contenido = word_tokenize(contenido)
    contenido = elimina_no_alfanumerico(contenido)
    contenido = elimina_palabras_vacias(contenido)
    contenido = reduce_a_raiz(contenido)
    return contenido

In [ ]:
pprint(procesa_mensaje(contenidos_mensajes_entrenamiento[-21]),
       compact=True)

In [ ]:
filtro_antispam = Pipeline([
    ('vectorizador', TfidfVectorizer(analyzer=procesa_mensaje)),
    ('modelo', KNeighborsClassifier(n_neighbors=4, metric='cosine'))
])

In [ ]:
filtro_antispam.fit(contenidos_mensajes_entrenamiento,
                    clases_mensajes_entrenamiento)

In [ ]:
predicciones_mensajes_prueba = filtro_antispam.predict(
    contenidos_mensajes_prueba
)
recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

### Ejercicio 2

En el cuaderno NLTK.ipynb se ha construido un sistema de predicción de texto en español basado en modelos de $n$-gramas. Estos modelos se han entrenado a partir de un corpus de textos en español que se ha usado en bruto. El objetivo de este ejercicio es recrear la construcción del sistema de predicción de texto, pero usando una versión normalizada del corpus.

#### Apartado 1

En este apartado se pide:

1. Leer el corpus guardado en el fichero `Texto predictivo/corpus_InfoLibros_parcial.txt` y dividirlo en un corpus de entrenamiento y un corpus de prueba.
2. Construir modelos unigramas, bigramas y trigramas, con y sin suavizado, a partir del corpus de entrenamiento normalizado convirtiendo todas las palabras a minúsculas.
3. Seleccionar el modelo con menor perplejidad sobre el corpus de prueba normalizado convirtiendo todas las palabras a minúsculas.

In [ ]:
# Nos aseguramos de haber descargado el tokenizador

from nltk import download

download('punkt', download_dir='.')

In [ ]:
from nltk.corpus.reader.plaintext import PlaintextCorpusReader
from nltk.data import load

In [ ]:
corpus_InfoLibros = PlaintextCorpusReader(
    root='Texto predictivo',
    fileids=['corpus_InfoLibros_parcial.txt'],
    encoding='utf8',
    sent_tokenizer=load('tokenizers/punkt/spanish.pickle')
)

In [ ]:
total_frases = len(corpus_InfoLibros.sents())
total_frases

In [ ]:
total_frases_entrenamiento = int(total_frases * .8)
total_frases_entrenamiento

In [ ]:
corpus_entrenamiento = corpus_InfoLibros.sents()[:total_frases_entrenamiento]

In [ ]:
corpus_prueba = corpus_InfoLibros.sents()[total_frases_entrenamiento:]

In [ ]:
from nltk.lm.vocabulary import Vocabulary
from nltk.lm.preprocessing import flatten

In [ ]:
vocabulario_palabras = Vocabulary(
    (palabra.lower()
     for palabra in flatten(corpus_entrenamiento)),  # lista de todas las palabras
    unk_cutoff=50  # mínimo número de ocurrencias
)

In [ ]:
vocabulario_palabras.lookup('hola')

In [ ]:
inicio_frase = '<s>'
fin_frase = '</s>'
vocabulario_palabras.update({inicio_frase: 50, fin_frase: 50})

In [ ]:
def delimita_frase(frase, n):
    return (['<s>'] * (n - 1) +
            [palabra.lower() for palabra in frase] +
            ['</s>'])

In [ ]:
from pprint import pprint

In [ ]:
primera_frase_entrenamiento = corpus_entrenamiento[0]
pprint(delimita_frase(primera_frase_entrenamiento, 1),
       compact=True)
pprint(delimita_frase(primera_frase_entrenamiento, 2),
       compact=True)
pprint(delimita_frase(primera_frase_entrenamiento, 3),
       compact=True)

In [ ]:
from nltk.util import ngrams, bigrams, trigrams
from nltk.lm import MLE, Laplace

In [ ]:
for ii in ngrams(delimita_frase(corpus_entrenamiento[0], 1), n=1):
    print(ii)

In [ ]:
aa = (ngrams(delimita_frase(frase, 1), n=1) for frase in corpus_entrenamiento)

In [ ]:
aaa = next(aa)

In [ ]:
next(aaa)

In [ ]:
modelo_unigrama_MLE = MLE(1, vocabulary=vocabulario_palabras)
modelo_unigrama_MLE.fit(ngrams(delimita_frase(frase, 1), n=1)
                        for frase in corpus_entrenamiento)
modelo_unigrama_MLE.perplexity(flatten(ngrams(delimita_frase(frase, 1), n=1)
                                       for frase in corpus_prueba))

In [ ]:
modelo_unigrama_Laplace = Laplace(1, vocabulary=vocabulario_palabras)
modelo_unigrama_Laplace.fit(ngrams(delimita_frase(frase, 1), n=1)
                            for frase in corpus_entrenamiento)
modelo_unigrama_Laplace.perplexity(flatten(ngrams(delimita_frase(frase, 1), n=1)
                                           for frase in corpus_prueba))

In [ ]:
modelo_bigrama_MLE = MLE(2, vocabulary=vocabulario_palabras)
modelo_bigrama_MLE.fit(bigrams(delimita_frase(frase, 2))
                       for frase in corpus_entrenamiento)
modelo_bigrama_MLE.perplexity(flatten(bigrams(delimita_frase(frase, 2))
                                      for frase in corpus_prueba))

In [ ]:
modelo_bigrama_Laplace = Laplace(2, vocabulary=vocabulario_palabras)
modelo_bigrama_Laplace.fit(bigrams(delimita_frase(frase, 2))
                           for frase in corpus_entrenamiento)
modelo_bigrama_Laplace.perplexity(flatten(bigrams(delimita_frase(frase, 2))
                                          for frase in corpus_prueba))

In [ ]:
modelo_trigrama_MLE = MLE(3, vocabulary=vocabulario_palabras)
modelo_trigrama_MLE.fit(trigrams(delimita_frase(frase, 3))
                        for frase in corpus_entrenamiento)
modelo_trigrama_MLE.perplexity(flatten(trigrams(delimita_frase(frase, 3))
                                       for frase in corpus_prueba))

In [ ]:
modelo_trigrama_Laplace = Laplace(3, vocabulary=vocabulario_palabras)
modelo_trigrama_Laplace.fit(trigrams(delimita_frase(frase, 3))
                            for frase in corpus_entrenamiento)
modelo_trigrama_Laplace.perplexity(flatten(trigrams(delimita_frase(frase, 3))
                                           for frase in corpus_prueba))

#### Apartado 2

Definir una función `predice_palabras` que prediga, a partir de las palabras anteriores y de las letras de la palabra ya escritas, qué palabra se pretende escribir. La función debe actuar como sigue:

* Si todas las letras del prefijo escrito están en minúsculas, entonces debe predecir palabras en minúsculas.
* Si todas las letras del prefijo escrito están en mayúsculas, entonces debe predecir palabras en mayúsculas.
* Si el prefijo escrito mezcla letras en minúsculas y en mayúsculas, entonces:
  * Si la primera letra del prefijo está en minúsculas, entonces debe predecir palabras en minúsculas.
  * Si la primera letra del prefijo está en mayúsculas, entonces debe predecir palabras con la primera letra en mayúsculas y el resto en minúsculas.

In [ ]:
def predice_palabras(prefijo, contexto, numero_palabras, modelo):
    def puntua_palabra(palabra):
        return modelo.score(
            palabra,
            tuple(palabra_anterior.lower()
                  for palabra_anterior in contexto)
        )
    palabras_candidatas = filter(
        lambda palabra: palabra.startswith(prefijo.lower()),
        vocabulario_palabras
    )
    palabras_candidatas_ordenadas = sorted(palabras_candidatas,
                                           key=puntua_palabra,
                                           reverse=True)
    palabras_predichas = palabras_candidatas_ordenadas[:numero_palabras]
    if prefijo.islower():
        return palabras_predichas
    elif prefijo.isupper():
        return [palabra_predicha.upper()
                for palabra_predicha in palabras_predichas]
    elif prefijo[0].islower():
        return palabras_predichas
    else:
        return [palabra_predicha.capitalize()
                for palabra_predicha in palabras_predichas]

In [ ]:
predice_palabras('nat', ('Lenguaje',), 5, modelo_bigrama_Laplace)

In [ ]:
predice_palabras('NAT', ('Lenguaje',), 5, modelo_bigrama_Laplace)

In [ ]:
predice_palabras('nAt', ('Lenguaje',), 5, modelo_bigrama_Laplace)

In [ ]:
predice_palabras('NaT', ('Lenguaje',), 5, modelo_bigrama_Laplace)